# LowBitSparse Colab 验证手册

用于在 Colab A100 上复现 M0/M1/M2/M3: FP16 基线、RTN/GPTQ/AWQ 权重量化、稀疏注意力长序列基准、量化感知蒸馏、WikiText-2 PPL、理论压缩比和汇总表。


## 推荐执行顺序

1. 环境检查和依赖安装。
2. 本地逻辑验证: `pytest` + `cpu_smoke.py`。
3. 快速单点验证: RTN INT4 只评少量样本。
4. M1 核心验收: baseline + INT8 + INT4 + GPTQ + AWQ。
5. M2 基线验收: Sliding Window + StreamingLLM 长序列基准。
6. M2-c/M2-d/M2-e 后续: KV prune、chunked prefill 与 ring-buffer + CUDA graph decode。
7. M3 蒸馏验收: smoke + 正式 distill 训练。
8. 可选完整消融: `scripts/run_sweep.py`。

In [ ]:
# 1) 确认 GPU。M1 正式评测建议使用 A100。
!nvidia-smi -L

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
# 2) 挂载 Google Drive。结果写在 Drive 项目目录下,可避免 Colab 断连丢失。
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3) 获取代码。若 Drive 中已存在仓库,直接进入;否则 clone 一份。
%cd /content/drive/MyDrive/LowBitSparse/

如果你确认 Drive 中没有未提交改动,可以手动执行下一格更新代码。若 `git status --short` 有本地改动,先不要 pull。

In [ ]:
# 可选:更新仓库。若有本地改动导致失败,先保留现场再处理。
# !git pull --ff-only

In [ ]:
# 4) 安装依赖。Colab 自带 torch,requirements 只做宽松约束。
%pip install -q -r requirements.txt

import torch, transformers, datasets, accelerate
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)

In [ ]:
# 5) 可选:从 Colab Secrets 读取 Hugging Face token。
# 公开模型通常不需要 token;若遇到下载限流或私有模型,在 Secrets 中添加 HF_TOKEN。
import os
from google.colab import userdata

try:
    token = userdata.get("HF_TOKEN")
except Exception:
    token = None

if token:
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    print("HF_TOKEN 已设置")
else:
    print("未设置 HF_TOKEN,将按公开下载流程执行")

## 本地逻辑验证

这一步不下载 Qwen,用于确认量化数学、校准 hook、Linear 替换和压缩统计没有被改坏。

In [ ]:
!python -m pytest -q
!python scripts/cpu_smoke.py

## 快速单点验证

只跑 RTN INT4,并把 PPL 评测限制到 4 个窗口。适合验证 Colab 环境、模型下载、量化和结果落盘。

In [ ]:
# 生成一个临时 smoke config,不修改正式配置文件。
import copy
import yaml
from pathlib import Path

base = yaml.safe_load(Path("configs/qwen0.5b_int4.yaml").read_text(encoding="utf-8"))
smoke = copy.deepcopy(base)
smoke["exp_id"] = "m1_rtn_int4_g128_smoke"
smoke["eval"]["max_samples"] = 4
Path("configs/_colab_smoke_int4.yaml").write_text(
    yaml.safe_dump(smoke, allow_unicode=True, sort_keys=False),
    encoding="utf-8",
)
print(Path("configs/_colab_smoke_int4.yaml").read_text(encoding="utf-8"))

In [ ]:
!python main.py quant --config configs/_colab_smoke_int4.yaml

## M1 核心验收

这组命令生成 M1 最关键的对比: FP16 baseline、RTN INT8、RTN INT4、GPTQ INT4、AWQ INT4。

In [ ]:
!python main.py eval  --config configs/qwen0.5b_base.yaml
!python main.py quant --config configs/qwen0.5b_int8.yaml
!python main.py quant --config configs/qwen0.5b_int4.yaml
!python main.py quant --config configs/qwen0.5b_gptq_int4.yaml
!python main.py quant --config configs/qwen0.5b_awq_int4.yaml

!python scripts/summarize.py --results results --out results/m1_summary.md

In [ ]:
!python main.py quant --config configs/qwen0.5b_gptq_int4_embint8.yaml
!python main.py quant --config configs/qwen0.5b_gptq_int4_embint4.yaml

In [ ]:
# 查看汇总表。重点看 PPL、Delta PPL、压缩比和等效 bit。
from pathlib import Path
print(Path("results/m1_summary.md").read_text(encoding="utf-8"))

## 可选:完整 M1 消融

`run_sweep.py` 会扫描 method × bit × group_size。耗时明显长于核心验收,适合最终补表。先用 `--smoke` 确认流程,再打开完整 sweep。

In [ ]:
# 可选:全组合冒烟。每组只评 4 个 PPL 窗口,但 GPTQ/AWQ 仍会做校准。
# !python scripts/run_sweep.py --smoke
# !python scripts/summarize.py --results results --out results/m1_summary.md

In [ ]:
# 可选:完整消融。确认时间和额度足够后再执行。
RUN_FULL_SWEEP = True

if RUN_FULL_SWEEP:
    !python scripts/run_sweep.py
    !python scripts/summarize.py --results results --out results/m1_summary.md
else:
    print("RUN_FULL_SWEEP=False,已跳过完整消融")

## 补跑 GPTQ/AWQ per-channel。

重新汇总,新增的 m1_gptq_int4_gpc / m1_awq_int4_gpc 会并入表格。

In [ ]:
# 补跑 GPTQ/AWQ per-channel。复用 run_sweep.run_one,exp_id 自动为 m1_{method}_int4_gpc。
import sys, traceback
sys.path.insert(0, "scripts")
from scripts.run_sweep import run_one
from lowbitsparse.utils import load_config

base_cfg = load_config("configs/qwen0.5b_int4.yaml")
for method in ("gptq", "awq"):
    try:
        run_one(base_cfg, method, n_bits=4, group_size=-1, sym=False, smoke=False)
    except Exception:
        print(f"[失败] {method} per-channel:\n{traceback.format_exc()}")

# 重新汇总,新增的 m1_gptq_int4_gpc / m1_awq_int4_gpc 会并入表格。
!python scripts/summarize.py --results results --out results/m1_summary.md
from pathlib import Path
print(Path("results/m1_summary.md").read_text(encoding="utf-8"))

## 重跑 AWQ:恢复无裁剪好值(M1-h 负结果)

AWQ 的裁剪搜索(auto_clip)经 A100 实测**全面恶化 PPL**(INT3 +17),已默认关闭(见 OPTIMIZATION M1-h)。当前 `results/m1_awq_*.json` 是裁剪的坏值,下一格用默认(裁剪关)重跑 5 个 AWQ 组合恢复正确结果。

In [ ]:
# 重跑 AWQ 全组合(裁剪现默认关),覆盖裁剪坏值,恢复无裁剪好值
!python scripts/run_sweep.py --only awq --no-emb
!python scripts/summarize.py --results results --out results/m1_summary.md

from pathlib import Path
print(Path("results/m1_summary.md").read_text(encoding="utf-8"))

## M2 稀疏注意力

这组命令跑 Sliding Window 和 StreamingLLM 的长序列对照。`block_sparse` 作为可选实验保留在配置里，确认时间和额度足够再开。


In [ ]:
# M2 核心验收: baseline vs sparse 长序列基准。
!python main.py sparse --config configs/qwen0.5b_sparse_sliding.yaml
!python main.py sparse --config configs/qwen0.5b_sparse_streaming.yaml

# 可选:块稀疏
!python main.py sparse --config configs/qwen0.5b_sparse_block.yaml


## M2-c StreamingLLM KV cache 裁剪

这一步在 M2-b 的 StreamingLLM 上做真实 KV cache 裁剪。质量仍看 mask 参考,decode 阶段则走 sink + window 的短 cache 路径。


In [ ]:
# M2-c:StreamingLLM KV cache pruning.
!python main.py sparse --config configs/qwen0.5b_sparse_streaming_kvprune.yaml


In [ ]:
!python main.py sparse --config configs/qwen0.5b_sparse_streaming_chunked.yaml

## M2-d chunked prefill / local attention

这一步把 prefill 拆成 query chunk,chunk 之间用 StreamingLLM 规则裁剪 KV cache。质量仍用 additive mask 参考;latency/memory 走真实 chunked prefill + KV prune 路径。

In [ ]:
# M2-d:StreamingLLM chunked prefill / local attention.
!python main.py sparse --config configs/qwen0.5b_sparse_streaming_chunked.yaml

## M2-e ring-buffer + CUDA graph decode

这一步验证有界 `RingKVCache` + CUDA graph replay。latency/memory 走真实 ring+graph 路径;质量仍用 StreamingLLM additive mask 的 teacher-forced PPL 参考。当前是 benchmark proof,还不是可直接替换 `generate()` 的生产实现。


In [ ]:
# 步骤 1:CPU 单测(验证回绕逻辑)
# !python -m pytest tests/test_sparse.py -q -k ring

# 步骤 2:先行门控(隔离验证 RingKVCache + graph 不崩)
# !python scripts/cudagraph_probe.py --ring --prefill 2048 --decode 128

# 步骤 3:门控过了再跑集成 benchmark
!python main.py sparse --config configs/qwen0.5b_sparse_streaming_ringgraph.yaml


In [ ]:
# 查看 M2/M2-c/M2-d/M2-e 结果文件和逐长度对照。
import json
from pathlib import Path

patterns = ("m2_*.json", "m2c_*.json", "m2d_*.json", "m2e_*.json")
paths = sorted({p for pat in patterns for p in Path("results").glob(pat)})

for path in paths:
    data = json.loads(path.read_text(encoding="utf-8"))
    print(f"== {path.name} ==")
    if data.get("quality_note"):
        print("note=", data["quality_note"])
    for row in data.get("rows", []):
        baseline_ppl = row.get("baseline", {}).get("ppl", {}).get("ppl")
        sparse_ppl = row.get("sparse", {}).get("ppl", {}).get("ppl")
        mem_delta = row.get("memory_delta_mb")
        print(
            row.get("seqlen"),
            "prefill_speedup=", row.get("speedup", {}).get("prefill"),
            "decode_speedup=", row.get("speedup", {}).get("decode"),
            "mem_delta_mb=", mem_delta,
            "baseline_ppl=", baseline_ppl,
            "sparse_ppl=", sparse_ppl,
        )


## M3 量化感知蒸馏

这一步先生成一个 smoke config 再跑正式 distill。smoke 只为确认 teacher / student / loss / save_results 链路，正式配置会把结果写到 `results/m3_distill_qwen0.5b.json`。


In [ ]:
# 生成一个临时 M3 smoke config,不修改正式配置文件。
import copy
import yaml
from pathlib import Path

base = yaml.safe_load(Path("configs/qwen0.5b_distill.yaml").read_text(encoding="utf-8"))
smoke = copy.deepcopy(base)
smoke["exp_id"] = "m3_distill_qwen0.5b_smoke"
smoke["distill"]["train_samples"] = 32
smoke["distill"]["eval_samples"] = 8
smoke["distill"]["max_steps"] = 10
smoke["distill"]["eval_every"] = 5
smoke["distill"]["log_every"] = 2
smoke["distill"]["save_student_path"] = "results/m3_distill_smoke.pt"
Path("configs/_colab_smoke_distill.yaml").write_text(
    yaml.safe_dump(smoke, allow_unicode=True, sort_keys=False),
    encoding="utf-8",
)
print(Path("configs/_colab_smoke_distill.yaml").read_text(encoding="utf-8"))

In [ ]:
# 先跑小步数 smoke，确认链路正常。
!python main.py distill --config configs/_colab_smoke_distill.yaml

In [ ]:
# 正式 M3 蒸馏。
!python main.py distill --config configs/qwen0.5b_distill.yaml

In [ ]:
!python scripts/run_m3_ablation.py --modes scale lora --loss-grid 0.7:0.3
!python scripts/build_m4_report.py


In [ ]:
# 查看 M3 结果。
from pathlib import Path
print(Path("results/m3_distill_qwen0.5b.json").read_text(encoding="utf-8"))

In [ ]:
#补跑 1.5B 关键复现

# 1.5B FP16 baseline
!python main.py eval --config configs/qwen1.5b_base.yaml

# 1.5B 推荐量化点: GPTQ INT4 + embedding INT8
!python main.py quant --config configs/qwen1.5b_gptq_int4_embint8.yaml

# 1.5B 长序列 decode: M2-e ring-buffer + CUDA graph
!python main.py sparse --config configs/qwen1.5b_sparse_streaming_ringgraph.yaml

# 重新生成 M4 总表和报告
!python scripts/build_m4_report.py

In [18]:
!python scripts/run_m3_ablation.py --modes lora --loss-grid 0.7:0.3
!python scripts/build_m4_report.py

[M3-ablation] run: m3_ablate_lora_a07_b03
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 290/290 [00:00<00:00, 1868.58it/s]
Loading weights: 100% 290/290 [00:00<00:00, 1781.62it/s]
[08:51:10] INFO lowbitsparse: [M3] student prepared: replaced=168, mode=lora, trainable=4399104 (0.8826%), quant=QuantConfig(n_bits=4, group_size=128, symmetric=False, method='rtn', skip=['lm_head'], calib_n_samples=128, calib_seqlen=512, quant_embedding=False, embedding_bits=None)
[08:51:17] INFO lowbitsparse: [M3] step=1 loss=181.1795 kd=257.6160 ce=2.8277
[08:51:19] INFO lowbitsparse: [M3] step=10 loss=164.1920 kd=233.1883 ce=3.2008
[08:51:22] INFO lowbitsparse: [M3] step=20 loss=171.2460 kd=243.3135 ce=3.0883
[08:51:28] INFO lowbitsparse: [M3] step=30 loss=170.7185 kd=242.5837 ce=3.0331
[08:51:30] INFO lowbitsparse: [M3] step=40 loss=151.1057 kd=214.6868 ce=2.7496
[08:51:37] INFO lowbitsparse: [M3] step=50 loss=152.1169 kd=216.0435 ce=2.9550
[08:51:40] INFO lowbits

## 结果备份

如果项目目录已经在 Drive 中,结果天然保存在 Drive。下面这格会额外复制一份带时间戳的 results 快照,避免覆盖历史实验。

In [ ]:
import shutil
from datetime import datetime
from pathlib import Path

src = Path("results").resolve()
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
dst = Path("/content/drive/MyDrive/LowBitSparse_results") / stamp
dst.parent.mkdir(parents=True, exist_ok=True)

if src.exists():
    shutil.copytree(src, dst)
    print("results 快照已保存到:", dst)
else:
    print("results 目录不存在,没有可备份内容")